# تشخیص پلاک خودرو (Vehicle + License Plate Detection + OCR)

خط لوله:
1. **YOLO** برای تشخیص وسیله‌ی نقلیه و نوع آن (Car / Motorcycle / Bus / Truck / ...)
2. مدل **YOLOv8 پلاک** برای پیدا کردن پلاک داخل هر خودرو
3. **OCR**: پلاک فارسی → **Hezar CRNN**، پلاک لاتین → **fast-plate-ocr** (یا **PaddleOCR** در حالت ENGLISH؛ fallback همه: EasyOCR)
4. تحلیل کشوری بر اساس `COUNTRY_MODE` + ردیابی ساده تا متن پلاک روی ویدیو پرش نکند

حالت‌های کشور:
- `IR` — تحلیل کامل پلاک ایران: ارقام/حرف، کد استان، رنگ، نوع پلاک، منطقه آزاد
- `GLOBAL` — فقط تشخیص + متن لاتین
- `AUTO` — برای هر پلاک خودکار: پلاک ایرانیِ معتبر → تحلیل ایران، وگرنه → لاتین
- `ENGLISH` — مثل GLOBAL ولی با موتور OCR اختصاصیِ انگلیسی (PaddleOCR)، برای فونت/متن انگلیسی دقت بهتری می‌دهد

ساختار نوت‌بوک:
- بخش ۱: نصب و ایمپورت
- بخش ۲: تنظیمات (مسیرها، حالت کشور، وسایل نقلیه، پارامترها)
- بخش ۳: انتخاب خودکار GPU/CPU و بارگذاری مدل‌ها
- بخش ۴: توابع کمکی (OCR، رندر متن فارسی روی ویدیو)
- بخش ۵: داده‌ها و منطق پلاک ایران (کد استان، رنگ، نوع)
- بخش ۶: تابع اصلی پردازش ویدیو (ردیابی + اجماع کاراکتربه‌کاراکتر بین فریم‌ها)
- بخش ۷: تست روی ویدیوها + ذخیره و نمایش خروجی
- بخش ۸: ساخت GIF از خروجی

---

## تنظیم دقت/سرعت بر اساس دوربین (بدون دست‌زدن به کد)

منطق تشخیص و OCR این نوت‌بوک ثابت است؛ **فقط دو پارامتر دستی** از یک فایل کانفیگ خوانده می‌شوند، و یک پارامتر سوم به‌طور خودکار از روی آن‌ها محاسبه می‌شود — پس می‌توانید بدون لمس یک خط کد، پروفایل را برای هر دوربین/صحنه عوض کنید:

- **مدل خودرو** (دستی): کدام YOLO برای تشخیص خودرو استفاده شود (سبک‌تر/سریع‌تر در برابر سنگین‌تر/دقیق‌تر)
- **imgsz خودرو** (دستی): اندازه‌ی تصویر ورودی — این مقدار **فقط** به مدل تشخیص خودرو داده می‌شود، نه به مدل پلاک
- **OCR_MIN_H** (خودکار، محاسبه‌شده) = `64 × imgsz/640`: هرچه `imgsz` بزرگ‌تر باشد، ماشین‌های دورتر/کوچک‌تری هم دیده می‌شوند که پلاکشان در پیکسل واقعی خیلی کوچک است؛ این پارامتر آستانه‌ی بزرگ‌نمایی *قبل از OCR* را متناسب بزرگ می‌کند تا کیفیت OCR روی این پلاک‌های ریز افت نکند.

این پارامترها از فایل `PelakX/configs/pelak3.yaml` خوانده می‌شوند. ۴ پروفایل آماده در `PelakX/configs/`:

| فایل | imgsz | OCR_MIN_H (خودکار) | مناسب برای |
|---|---|---|---|
| `baseline_640.yaml` | 640 | 64 | رفتار پایه‌ی این نوت‌بوک (برای تست رگرسیون) |
| `pelak3.yaml` (پیش‌فرض) | 960 | 96 | میانه — هم خلوت هم بزرگراه قابل‌قبول |
| `highway.yaml` | 1280 | 128 | بزرگراه |
| `highway_far.yaml` | 1600 | 160 | بزرگراه/دوربین خیلی دور |

برای عوض‌کردن پروفایل، مقدار `CONFIG_PATH` را در سلول تنظیمات به یکی از این فایل‌ها بدهید. اگر فایل کانفیگ پیدا نشود، رفتار پایه (۶۴۰) استفاده می‌شود.

### چند نکته‌ی مهندسی که هنگام تنظیم این پارامترها یاد گرفتیم

1. **`imgsz` را فقط به مدل تشخیص خودرو بدهید، نه به مدل پلاک.** مدل پلاک با اندازه‌ی پیش‌فرض کار می‌کند؛ عوض کردنش جعبه‌ی پلاک را کمی جابه‌جا می‌کند و همان تفاوت کوچک مستقیم به دقت OCR ضربه می‌زند. لِوِر درست برای «هوشمند کردن» crop، بزرگ‌نمایی *بعد از* پیدا شدن پلاک بود (`OCR_MIN_H`)، نه تغییر خود تشخیص.
2. **هر فریم را پردازش کنید، فریم رد نکنید.** رد کردن فریم‌ها برای صرفه‌جویی در محاسبات باعث می‌شود جعبه/برچسب پلاک هر فریم دوم محو شود و خروجی خاموش‌روشن بزند — تجربه‌ی کاربری را بیشتر از هزینه‌ی محاسباتی که صرفه‌جویی می‌کند خراب می‌کند.

**نکته:** محدودیت نرخ OCR (مثل «هر ترک حداکثر N بار در ثانیه») در این نوت‌بوک پیاده‌سازی نشده — همه‌چیز ساده و مستقیم است.

---

## محدودیت‌های شناخته‌شده و همکاری

این پروژه یک نمونه‌ی شخصی/آموزشی است و تا این مرحله عمداً ساده نگه داشته شده. چند محدودیت که خودمان می‌دانیم:

- **پلاک‌های خیلی کوچک/دور** (ارتفاع زیر ~۲۰ پیکسل): OCR ناپایدار است؛ اجماع بین فریم‌ها کمک می‌کند ولی معجزه نمی‌کند. رزولوشن دوربین مهم‌تر از انتخاب مدل است.
- **پلاک‌های ۱۰ کاراکتری** (مثلاً هند): مدل لاتین حداکثر ۱۰ اسلات دارد و در فاصله‌ی دور معمولاً کاراکتر آخر را می‌اندازد.
- **اشتباهات شبیه‌نویسه** (O/0، D/0، I/1): بدون دانستن قالب پلاک هر کشور به‌طور قطعی حل نمی‌شود.
- **ردیابی** ساده و فاصله‌محور است، نه یک tracker کامل (ByteTrack/DeepSORT).
- **حالت `ENGLISH`** به PaddleOCR نیاز دارد که باید جداگانه نصب شود؛ اگر نبود، خودکار به fast-plate-ocr برمی‌گردد.

اگر به یک نسخه‌ی کامل‌تر (دیتاست اختصاصی، fine-tune مدل پلاک/OCR، tracker حرفه‌ای، اجرای real-time) نیاز دارید، خوشحال می‌شوم همکاری کنیم — از طریق Issues همین مخزن پیام بدهید.


## بخش ۱ — نصب و ایمپورت

In [32]:
# اگر از کرنل «Python (pelak)» استفاده می‌کنی این سلول لازم نیست (کامنت بماند).
%pip install -q opencv-python ultralytics easyocr numpy torch torchvision
# OCR فارسی + رندر متن RTL روی ویدیو
%pip install -q hezar arabic-reshaper python-bidi
# OCR دقیق پلاک لاتین (۶۵+ کشور)
%pip install -q fast-plate-ocr onnxruntime
# OCR انگلیسی (حالت ENGLISH) — فونت لاتین، دقت بالا
%pip install -q paddlepaddle paddleocr

In [33]:
import csv
import urllib.request
from collections import Counter
from pathlib import Path

import cv2
import numpy as np
import torch
from ultralytics import YOLO
import easyocr

## بخش ۲ — تنظیمات

همه‌ی پارامترها و مسیرها فقط همین‌جا.

In [36]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import drive
import yaml
from pathlib import Path

drive.mount('/content/drive')


VIDEO_DIR = Path("/content/drive/MyDrive")

OUTPUT_DIR = Path("/content/output")
MODELS_DIR = Path("/content/models")

# check videos folder 
if not VIDEO_DIR.exists():
    print(f"⚠️ Error : not founde video folder {VIDEO_DIR}")
    VIDEO_DIR.mkdir(parents=True, exist_ok=True)

# OR in your computer : 

# # --- Paths (absolute) ---
# PROJECT_DIR = Path(r"C:\1\1_پروژه\تشخیص پلاکو ماشین")
# VIDEO_DIR = PROJECT_DIR / "ویدیو"      # input videos folder (kept as the real folder name on disk)
# OUTPUT_DIR = PROJECT_DIR / "خروجی"     # outputs are written here (kept as the real folder name on disk)
# MODELS_DIR = PROJECT_DIR / "models"
# OUTPUT_DIR.mkdir(exist_ok=True)
# MODELS_DIR.mkdir(exist_ok=True)

# # check videos folder 
# if not VIDEO_DIR.exists():
#     print(f"⚠️ Error : not founde video folder {VIDEO_DIR}")
#     VIDEO_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:

# --- تنظیمات مدل و کانفیگ ---
# اگر فایل کانفیگی در درایو دارید مسیر را اینجا بدهید، در غیر این صورت تنظیمات پیش‌فرض اعمال می‌شود
CONFIG_PATH = Path("/content/drive/MyDrive/pelak3.yaml")

_cfg = {}
if CONFIG_PATH.exists():
    _cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8")) or {}
    print(f"⚙️ کانفیگ بارگذاری شد: {CONFIG_PATH.name}")
else:
    print(f"⚠️ فایل کانفیگ اختصاصی پیدا نشد، از مقادیر پیش‌فرض استفاده می‌شود.")

_wanted_model = _cfg.get("vehicle_model", "yolov8s.pt")
VEHICLE_MODEL_PATH = _wanted_model

PLATE_MODEL_PATH = MODELS_DIR / "license_plate_model.pt"
PLATE_MODEL_URL = (
    "https://github.com/Muhammad-Zeerak-Khan/"
    "Automatic-License-Plate-Recognition-using-YOLOv8/raw/main/license_plate_detector.pt"
)

# --- سایر تنظیمات ---
COUNTRY_MODE = "GLOBAL" # می توانید به "IR" تغییر دهید
SUPPORTED_COUNTRIES = {
    "IR": "ایران 🇮🇷", "AE": "امارات 🇦🇪", "TR": "ترکیه 🇹🇷", "US": "آمریکا 🇺🇸",
}

VEHICLE_NAMES = {1: "Bicycle", 2: "Car", 3: "Motorcycle", 5: "Bus", 6: "Train", 7: "Truck"}
VEHICLE_CLASSES = list(VEHICLE_NAMES)

IR_LOGIC = COUNTRY_MODE in ("IR", "AUTO")
OCR_LANGS = ["fa", "en"] if IR_LOGIC else ["en"]

VEHICLE_CONF = 0.5
PLATE_CONF = 0.4
PROGRESS_EVERY = 30

IMGSZ = int(_cfg.get("imgsz", 960))
OCR_MIN_H = round(64 * IMGSZ / 640)

print(f"✅ مسیر ورودی (درایو): {VIDEO_DIR}")
print(f"✅ مسیر خروجی (کولب): {OUTPUT_DIR}")
print(f"✅ مدل خودرو: {VEHICLE_MODEL_PATH} | imgsz={IMGSZ}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⚠️ فایل کانفیگ اختصاصی پیدا نشد، از مقادیر پیش‌فرض استفاده می‌شود.
✅ مسیر ورودی (درایو): /content/drive/MyDrive
✅ مسیر خروجی (کولب): /content/output
✅ مدل خودرو: yolov8s.pt | imgsz=960


## بخش ۳ — انتخاب خودکار GPU/CPU و بارگذاری مدل‌ها

In [37]:
# انتخاب خودکار: اگر کارت گرافیک CUDA موجود باشد → GPU، وگرنه → CPU
USE_GPU = torch.cuda.is_available()
DEVICE = 0 if USE_GPU else "cpu"
print(f"🖥️ دستگاه: {'GPU (' + torch.cuda.get_device_name(0) + ')' if USE_GPU else 'CPU'}")

🖥️ دستگاه: GPU (Tesla T4)


In [38]:
# دانلود مدل پلاک (فقط بار اول)
if not PLATE_MODEL_PATH.exists():
    print("📥 در حال دریافت مدل پلاک از گیت‌هاب...")
    req = urllib.request.Request(PLATE_MODEL_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp, open(PLATE_MODEL_PATH, "wb") as f:
        f.write(resp.read())
    print(f"✅ ذخیره شد: {PLATE_MODEL_PATH}")
else:
    print(f"✅ مدل پلاک موجود است: {PLATE_MODEL_PATH}")

✅ مدل پلاک موجود است: /content/models/license_plate_model.pt


In [39]:
vehicle_model = YOLO(VEHICLE_MODEL_PATH)
plate_model = YOLO(str(PLATE_MODEL_PATH))

# --- OCR فارسی: Hezar CRNN (اگر لود نشد، EasyOCR) ---
easy_reader = easyocr.Reader(OCR_LANGS, gpu=USE_GPU)
hezar_model = None
if IR_LOGIC:
    try:
        from hezar.models import Model
        hezar_model = Model.load("hezarai/crnn-fa-license-plate-recognition-v2")
        print("✅ OCR فارسی: Hezar CRNN")
    except Exception as e:
        print(f"⚠️ Hezar لود نشد ({e}) — fallback: EasyOCR")

# --- OCR لاتین: fast-plate-ocr (مدل جهانی، بسیار دقیق برای پلاک انگلیسی) ---
latin_ocr = None
if COUNTRY_MODE in ("GLOBAL", "AUTO", "ENGLISH"):
    try:
        from fast_plate_ocr import LicensePlateRecognizer
        # cct-s = دقیق‌تر و کمی سنگین‌تر؛ cct-xs = سبک‌تر
        latin_ocr = LicensePlateRecognizer("cct-s-v2-global-model")
        print("✅ OCR لاتین: fast-plate-ocr (cct-s-v2-global)")
    except Exception as e:
        print(f"⚠️ fast-plate-ocr لود نشد ({e}) — fallback: EasyOCR")

# --- OCR انگلیسی اختصاصی: PaddleOCR (فقط حالت ENGLISH) ---
# برای متن/فونتِ لاتین به‌طور کلی (نه فقط قالب پلاک) دقت بالاتری از
# fast-plate-ocr می‌دهد؛ اگر لود نشد، به fast-plate-ocr و در آخر EasyOCR
# سقوط می‌کنیم (نگاه کن به _read_latin در بخش ۴).
paddle_ocr_en = None
if COUNTRY_MODE == "ENGLISH":
    try:
        from paddleocr import PaddleOCR
        paddle_ocr_en = PaddleOCR(use_angle_cls=True, lang="en", show_log=False)
        print("✅ OCR انگلیسی: PaddleOCR (en)")
    except Exception as e:
        print(f"⚠️ PaddleOCR لود نشد ({e}) — fallback: fast-plate-ocr/EasyOCR")

print(f"✅ مدل خودرو: {Path(VEHICLE_MODEL_PATH).name}  |  حالت: {COUNTRY_MODE}")

✅ OCR لاتین: fast-plate-ocr (cct-s-v2-global)
✅ مدل خودرو: yolov8s.pt  |  حالت: GLOBAL


## بخش ۴ — توابع کمکی

In [40]:
from PIL import Image, ImageDraw, ImageFont

try:
    import arabic_reshaper
    from bidi.algorithm import get_display
    _HAS_RTL = True
except Exception:
    _HAS_RTL = False

# فونتی که حروف فارسی دارد (Tahoma روی همه‌ی ویندوزها هست)
try:
    _FONT = ImageFont.truetype(r"C:\Windows\Fonts\tahoma.ttf", 32)
except Exception:
    _FONT = ImageFont.load_default()

_DIGITS_TO_ASCII = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")
_is_fa = lambda s: any("؀" <= c <= "ۿ" for c in s)


def _shape_fa(text):
    """آماده‌سازی متن فارسی برای رندر صحیح (اتصال حروف + راست‌به‌چپ)."""
    if _HAS_RTL and _is_fa(text):
        try:
            return get_display(arabic_reshaper.reshape(text))
        except Exception:
            return text
    return text


def draw_box(frame, x1, y1, x2, y2, color):
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)


def draw_plate_label(frame, x1, y1, text, color_bgr):
    """برچسب بالای پلاک را با Pillow می‌کشد تا فارسی/ارقام فارسی درست نشان داده شوند."""
    if not text:
        return
    disp = _shape_fa(str(text))
    img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    d = ImageDraw.Draw(img)
    l, t, r, b = d.textbbox((0, 0), disp, font=_FONT)
    tw, th = r - l, b - t
    ty = max(0, y1 - th - 15)
    fill = (color_bgr[2], color_bgr[1], color_bgr[0])
    d.rectangle([x1, ty, x1 + tw + 15, ty + th + 12], fill=fill)
    d.text((x1 + 8, ty + 4), disp, font=_FONT, fill=(0, 0, 0))
    frame[:] = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)


# ============================================================================
#  آماده‌سازی crop پلاک برای OCR
# ============================================================================
#  چیزی که واقعاً روی دقت OCR اثر دارد (بر اساس کانفیگ خود مدل‌ها + مقالات ALPR):
#    • fast-plate-ocr ورودی را به 128×64 و Hezar به 384×32 می‌برد، *بدون* حفظ
#      aspect ratio → بزرگ‌نمایی ما برای این دو بی‌اثر است؛ آنچه مهم است: crop
#      tight ولی با حاشیه‌ی کم (تا کاراکتر لبه بریده نشود)، و صاف بودن پلاک.
#    • fast-plate-ocr مدل RGB است؛ OpenCV BGR می‌دهد → باید تبدیل شود.
#    • CLAHE + فیلتر دوطرفه + unsharp زیر نور ناهموار/سایه کمک می‌کند؛
#      binarization به CRNN ضربه می‌زند (انجام نمی‌دهیم).
#    • deskew: پلاک زاویه‌دار را با تخمین زاویه‌ی خطوط افقی (Hough) صاف می‌کنیم.
#  چون هیچ پیش‌پردازشی روی *همه‌ی* شرایط برنده نیست، دو واریانت (خام و
#  تقویت‌شده) به OCR می‌دهیم و پراطمینان‌ترین را نگه می‌داریم.
# ============================================================================
PLATE_MARGIN_X = 0.08      # حاشیه‌ی افقی نسبت به عرض باکس (هر طرف)
PLATE_MARGIN_Y = 0.15      # حاشیه‌ی عمودی نسبت به ارتفاع باکس (هر طرف)
PLATE_MIN_W, PLATE_MIN_H = 24, 10   # کوچک‌تر از این، OCR فقط نویز تولید می‌کند
PLATE_GOOD_H = 32                   # از این ارتفاع به بالا، خوانش وزن کامل در رای‌گیری دارد
PLATE_MIN_SHARPNESS = 15.0          # واریانس لاپلاسین؛ کمتر از این = بلور شدید
DESKEW_MAX_ANGLE = 20.0             # زاویه‌های بزرگ‌تر خطای تخمین‌اند، نه کجی پلاک


def crop_plate(frame_clean, gbox):
    """باکس پلاک (مختصات کل فریم) → crop با حاشیه از فریمِ *بدون نقاشی*."""
    gx1, gy1, gx2, gy2 = gbox
    w, h = gx2 - gx1, gy2 - gy1
    if w < PLATE_MIN_W or h < PLATE_MIN_H:
        return None
    mx, my = int(w * PLATE_MARGIN_X), int(h * PLATE_MARGIN_Y)
    H, W = frame_clean.shape[:2]
    x1, y1 = max(0, gx1 - mx), max(0, gy1 - my)
    x2, y2 = min(W, gx2 + mx), min(H, gy2 + my)
    crop = frame_clean[y1:y2, x1:x2]
    return crop if crop.size else None


def _upscale(crop, min_h=OCR_MIN_H):
    """بزرگ‌نمایی پلاک‌های ریز — برای EasyOCR/PaddleOCR مؤثر است و تخمین زاویه و
    CLAHE را هم پایدارتر می‌کند (برای fast-plate/Hezar خنثی است)."""
    h, w = crop.shape[:2]
    if 0 < h < min_h:
        f = min_h / h
        crop = cv2.resize(crop, (int(w * f), min_h), interpolation=cv2.INTER_CUBIC)
    return crop


def _sharpness(gray):
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())


def _deskew(img):
    """زاویه‌ی غالب خطوط تقریباً افقی (لبه‌ی پلاک/خط پایه‌ی حروف) را با Hough
    می‌گیرد و تصویر را حول مرکز می‌چرخاند. زاویه‌های < 1° را دست نمی‌زند."""
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=max(20, w // 4),
                            minLineLength=w // 3, maxLineGap=10)
    if lines is None:
        return img, 0.0
    angles = []
    for x1, y1, x2, y2 in lines[:, 0]:
        a = float(np.degrees(np.arctan2(y2 - y1, x2 - x1)))
        if abs(a) <= DESKEW_MAX_ANGLE:
            angles.append(a)
    if not angles:
        return img, 0.0
    ang = float(np.median(angles))
    if abs(ang) < 1.0:
        return img, ang
    M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_CUBIC,
                          borderMode=cv2.BORDER_REPLICATE), ang


def _enhance(img):
    """CLAHE روی کانال روشنایی + فیلتر دوطرفه (نویز کم، لبه‌ها حفظ) + unsharp."""
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4)).apply(l)
    out = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)
    out = cv2.bilateralFilter(out, 5, 40, 40)
    blur = cv2.GaussianBlur(out, (0, 0), 1.0)
    return cv2.addWeighted(out, 1.5, blur, -0.5, 0)


def plate_variants(crop):
    """crop → لیست واریانت‌های BGR برای OCR؛ [] یعنی کیفیت آن‌قدر پایین است که
    OCR فقط نویز به رای‌گیری اضافه می‌کند."""
    if crop is None or crop.size == 0:
        return []
    if crop.ndim == 2:
        crop = cv2.cvtColor(crop, cv2.COLOR_GRAY2BGR)
    crop = _upscale(crop)
    if _sharpness(cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)) < PLATE_MIN_SHARPNESS:
        return []
    plain, _ = _deskew(crop)
    return [plain, _enhance(plain)]


# ---------- نرمال‌سازی متن (کلید رای‌گیری بین فریم‌ها) ----------
def _norm_latin(t):
    return "".join(c for c in t.upper() if c.isalnum())


def _norm_fa(t):
    return " ".join(t.split())


# ---------- موتورهای OCR (ورودی: یک واریانت BGR آماده) ----------
def _ocr_easy(img):
    res = easy_reader.readtext(img)
    if not res:
        return "", 0.0
    res.sort(key=lambda r: r[0][0][0])
    return " ".join(r[1] for r in res).strip(), float(np.mean([r[2] for r in res]))


def _ocr_hezar(img):
    out = hezar_model.predict(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    while isinstance(out, (list, tuple)) and out:
        out = out[0]
    if isinstance(out, dict):
        return str(out.get("text", "")).strip(), float(out.get("score", 0.9) or 0.9)
    if hasattr(out, "text"):
        return str(out.text).strip(), float(getattr(out, "score", 0.9) or 0.9)
    return (str(out).strip() if out is not None else ""), 0.9


def _ocr_fastplate(img):
    # مدل cct-*-v2-global طبق plate_config آن RGB است؛ آرایه‌ی numpy بدون تبدیل
    # وارد می‌شود، پس BGR→RGB اینجا الزامی است.
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    try:
        out = latin_ocr.run(rgb, return_confidence=True)
    except TypeError:
        out = latin_ocr.run(rgb)
    if isinstance(out, tuple) and len(out) >= 2:
        out = out[0]
    item = out[0] if isinstance(out, (list, tuple)) and out else out
    text = str(getattr(item, "plate", item if isinstance(item, str) else "") or "").replace("_", "").strip()
    probs = getattr(item, "char_probs", None)
    arr = np.asarray(probs, float).reshape(-1) if probs is not None else np.array([])
    conf = float(arr.mean()) if arr.size else 0.9
    return text, max(0.0, min(1.0, conf))


def _ocr_paddle_en(img):
    """PaddleOCR (حالت ENGLISH). crop پلاک خودش یک ناحیه‌ی متن است، پس اول فقط
    recognition (det=False) — روی crop کوچک، detector اغلب چیزی پیدا نمی‌کند."""
    def _flatten(res):
        lines = []
        for r in (res or []):
            if not r:
                continue
            for item in (r if isinstance(r, list) else [r]):
                if isinstance(item, (list, tuple)) and len(item) == 2 and isinstance(item[0], str):
                    lines.append((0, item[0], float(item[1])))          # det=False → (text, conf)
                elif isinstance(item, (list, tuple)) and len(item) == 2:
                    box, (t, c) = item                                  # det=True → (box, (text, conf))
                    lines.append((box[0][0], t, float(c)))
        return lines

    lines = _flatten(paddle_ocr_en.ocr(img, det=False, cls=True))
    if not lines or max(c for _, _, c in lines) < 0.5:
        lines = _flatten(paddle_ocr_en.ocr(img, det=True, cls=True)) or lines
    if not lines:
        return "", 0.0
    lines.sort(key=lambda r: r[0])
    return " ".join(t for _, t, _ in lines).strip(), float(np.mean([c for _, _, c in lines]))


def _read_fa(img):
    return _ocr_hezar(img) if hezar_model is not None else _ocr_easy(img)


def _read_latin(img):
    if COUNTRY_MODE == "ENGLISH" and paddle_ocr_en is not None:
        return _ocr_paddle_en(img)
    return _ocr_fastplate(img) if latin_ocr is not None else _ocr_easy(img)


def _read_variant(img):
    """یک واریانت آماده → (متن، اطمینان، script) طبق COUNTRY_MODE."""
    try:
        if COUNTRY_MODE == "IR":
            t, c = _read_fa(img)
            return t, c, "fa"
        if COUNTRY_MODE in ("GLOBAL", "ENGLISH"):
            t, c = _read_latin(img)
            return t, c, "latin"
        # --- AUTO: اول فارسی؛ فقط اگر پلاک ایرانیِ *معتبر* بود نگه می‌داریم ---
        t_fa, c_fa = _read_fa(img)
        if parse_iranian_plate(t_fa, img)["valid"]:
            return t_fa, c_fa, "fa"
        t_lat, c_lat = _read_latin(img)
        if t_lat:
            return t_lat, c_lat, "latin"
        return t_fa, c_fa, "fa"
    except Exception:
        t, c = _ocr_easy(img)
        return t, c, ("fa" if _is_fa(t) else "latin")


def read_plate(frame_clean, gbox):
    """باکس پلاک (مختصات کل فریم) → (متن نرمال‌شده، اطمینان، script، وزن).

    crop با حاشیه از فریم تمیز → گیت کیفیت → deskew → واریانت‌های خام/تقویت‌شده →
    OCR روی هر واریانت و انتخاب پراطمینان‌ترین. در حالت ENGLISH (PaddleOCR,
    سنگین‌تر) فقط واریانت تقویت‌شده خوانده می‌شود.

    وزن = اطمینان × (ارتفاع پلاک / PLATE_GOOD_H، سقف ۱): OCR روی پلاک ۱۲ پیکسلی
    هم گاهی با اطمینان بالا جواب *اشتباه* می‌دهد؛ بدون این ضریب، فریم‌های دور
    (که تعدادشان زیاد است) در رای‌گیری بر چند فریم نزدیک و واضح غلبه می‌کنند.
    """
    variants = plate_variants(crop_plate(frame_clean, gbox))
    if not variants:
        return "", 0.0, "latin", 0.0
    if COUNTRY_MODE == "ENGLISH" and paddle_ocr_en is not None:
        variants = variants[-1:]
    best = ("", 0.0, "latin")
    for v in variants:
        t, c, s = _read_variant(v)
        t = _norm_fa(t) if s == "fa" else _norm_latin(t)
        if t and c > best[1]:
            best = (t, c, s)
    h = gbox[3] - gbox[1]
    weight = best[1] * min(1.0, h / PLATE_GOOD_H)
    return best[0], best[1], best[2], weight

## بخش ۵ — داده‌ها و منطق پلاک ایران

منبع: `PelakX/configs/countries/ir.yaml` (کد استان‌ها از ghabzino + ویکی‌پدیا، معناشناسی حرف از نمونه‌های عکاسی‌شده).
در حالت `IR` روی همه‌ی پلاک‌ها اجرا می‌شود؛ در حالت `AUTO` فقط روی پلاک‌هایی که خط فارسی دارند.

In [41]:
# کد دورقمی استان (کدهای مشترک عیناً مطابق سیستم واقعی)
PROVINCE_CODES = {
    "10": "تهران", "11": "تهران", "12": "خراسان رضوی", "13": "اصفهان", "14": "خوزستان",
    "15": "آذربایجان شرقی", "16": "قم", "17": "آذربایجان غربی", "18": "همدان", "19": "کرمانشاه",
    "20": "تهران", "21": "البرز", "22": "تهران", "23": "اصفهان", "24": "خوزستان",
    "25": "آذربایجان شرقی", "26": "خراسان شمالی", "27": "آذربایجان غربی", "28": "همدان",
    "29": "کرمانشاه", "30": "تهران/البرز", "31": "لرستان", "32": "خراسان رضوی/شمالی/جنوبی",
    "33": "تهران", "34": "خوزستان", "35": "آذربایجان شرقی", "36": "خراسان رضوی",
    "37": "آذربایجان غربی", "38": "البرز", "39": "کرمانشاه", "40": "تهران", "41": "لرستان",
    "42": "خراسان رضوی", "43": "اصفهان", "44": "تهران", "45": "کرمان", "46": "گیلان",
    "47": "مرکزی", "48": "بوشهر", "49": "کهگیلویه و بویراحمد", "50": "تهران", "51": "کردستان",
    "52": "خراسان جنوبی", "53": "اصفهان", "54": "یزد", "55": "تهران", "56": "گیلان",
    "57": "مرکزی", "58": "بوشهر", "59": "گلستان", "60": "تهران", "61": "کردستان",
    "62": "مازندران", "63": "فارس", "64": "یزد", "65": "کرمان", "66": "تهران",
    "67": "اصفهان", "68": "البرز", "69": "گلستان", "71": "چهارمحال و بختیاری",
    "72": "مازندران", "73": "فارس", "74": "خراسان رضوی/شمالی", "75": "کرمان",
    "76": "گیلان", "77": "تهران", "78": "تهران/البرز", "79": "قزوین",
    "81": "چهارمحال و بختیاری", "82": "مازندران", "83": "فارس", "84": "هرمزگان",
    "85": "سیستان و بلوچستان", "86": "سمنان", "87": "زنجان", "88": "تهران", "89": "قزوین",
    "91": "اردبیل", "92": "مازندران", "93": "فارس", "94": "هرمزگان", "95": "سیستان و بلوچستان",
    "96": "سمنان", "97": "زنجان", "98": "ایلام", "99": "تهران",
}

# معنای حرف پلاک (نوع) + رنگ مورد انتظار زمینه
LETTER_SEMANTICS = {
    "الف": ("دولتی", "قرمز"), "پ": ("پلیس/انتظامی", "سبز"),
    "ت": ("تاکسی", "زرد"), "ع": ("حمل‌ونقل عمومی", "زرد"),
    "ک": ("ماشین‌آلات کشاورزی", "زرد"), "ژ": ("جانبازان و معلولین", "سفید"),
    "گ": ("گذر موقت", "سفید"), "ث": ("سپاه (تأیید‌نشده)", "?"),
    "D": ("دیپلمات", "آبی"), "S": ("سیاسی", "آبی"),
    "معلولین": ("جانبازان و معلولین", "سفید"),
    "تشریفات": ("تشریفات", "قرمز"), "موقت": ("موقت منطقه آزاد", "?"),
}

# حروف مجاز در جایگاه حرف پلاک شخصی/عمومی
IR_LETTERS = list("بپتثجحدزژسشصطعفقکگلمنوهی") + ["الف", "D", "S"]

FA_DIGITS = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")
NOISE_WORDS = ["ایران", "IRAN", "IR", "منطقه آزاد", "آزاد"]
print("✅ داده‌های پلاک ایران بارگذاری شد.")

✅ داده‌های پلاک ایران بارگذاری شد.


In [42]:
def detect_plate_color(plate_crop):
    """رنگ غالب زمینه‌ی پلاک را برمی‌گرداند: سفید/زرد/قرمز/سبز/آبی/نامشخص."""
    if plate_crop is None or plate_crop.size == 0:
        return "نامشخص"
    hsv = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2HSV)
    h, s, v = (int(np.median(hsv[:, :, i])) for i in range(3))
    if s < 60 and v > 120:
        return "سفید"
    if s < 60:
        return "نامشخص"
    if h < 12 or h > 168:
        return "قرمز"
    if 15 <= h <= 38:
        return "زرد"
    if 40 <= h <= 85:
        return "سبز"
    if 90 <= h <= 140:
        return "آبی"
    return "نامشخص"


def parse_iranian_plate(raw_text, plate_crop):
    """متن خام OCR + تصویر پلاک → دیکشنری تحلیل‌شده."""
    t = raw_text.translate(_DIGITS_TO_ASCII)
    for w in NOISE_WORDS:
        t = t.replace(w, " ")
    digits = "".join(c for c in t if c.isdigit())
    letters = (["الف"] if "الف" in t else []) + [c for c in t if c in IR_LETTERS]
    letter = letters[0] if letters else ""

    color = detect_plate_color(plate_crop)
    plate_type, _ = LETTER_SEMANTICS.get(letter, ("شخصی", "سفید"))
    if not letter and color == "زرد":
        plate_type = "عمومی/تاکسی (از روی رنگ)"

    out = {
        "raw": raw_text, "digits": digits, "letter": letter,
        "color": color, "type": plate_type, "province": "", "province_code": "",
        "free_zone": ("منطقه آزاد" in raw_text or "موقت" in raw_text),
        "formatted": raw_text, "valid": False,
    }

    if len(digits) >= 7 and letter:          # چیدمان شخصی/عمومی: DD L DDD + کد استان DD
        left, right, prov = digits[:2], digits[2:5], digits[5:7]
        out["province_code"] = prov
        out["province"] = PROVINCE_CODES.get(prov, "نامشخص")
        out["formatted"] = f"{left} {letter} {right} - ایران {prov}"
        out["valid"] = True
    elif len(digits) == 8 and not letter:    # موتورسیکلت
        out["type"] = "موتورسیکلت"
        out["province_code"] = digits[:3]
        out["formatted"] = f"{digits[:3]} - {digits[3:]}"
        out["valid"] = True

    return out


def ir_video_label(info):
    """برچسب روی ویدیو برای پلاک ایرانی (با Pillow رندر می‌شود، پس فارسی مشکلی ندارد)."""
    if info["valid"]:
        return info["formatted"]
    return info["digits"] or info["raw"] or "پلاک"


def latin_video_label(text):
    return text or "PLATE"

## بخش ۶ — تابع اصلی پردازش ویدیو

In [43]:
CSV_HEADER = ["track", "vehicle", "script", "raw", "digits", "letter", "color",
              "type", "province", "formatted", "conf", "frames_seen"]

# --- ردیابی ساده‌ی پلاک بین فریم‌ها ---
# آستانه‌ی «همان پلاک» متناسب با اندازه‌ی پلاک است: پلاک نزدیک (بزرگ) در پیکسل
# تندتر جابه‌جا می‌شود، پلاک دور (کوچک) کند. عدد ثابت بزرگ (مثل ۱۲۰ پیکسل روی
# ویدیوی ۸۴۸ پیکسلی) پلاک ماشینِ در حال عبور را به پلاک موتورِ پارک‌شده‌ی کناری
# می‌چسباند.
MATCH_DIST_MIN = 40        # کف آستانه (پیکسل)
MATCH_DIST_PLATE_W = 2.5   # آستانه = این ضریب × عرض پلاک
MATCH_MAX_GAP = 45         # ترکی که این تعداد فریم دیده نشده، دیگر ادامه داده نمی‌شود
MIN_FRAMES = 2             # پلاکی که کمتر از این تعداد فریم دیده شده، نویز فرض می‌شود

# اتوبوس/کامیون باکس خودرویی خیلی بزرگ‌تر از ماشین دارند ولی پلاکشان هم‌سایز
# پلاک ماشین است؛ اگر همان imgsz پیش‌فرض (۶۴۰) روی این crop بزرگ اعمال شود،
# پلاک نسبت به کل تصویر خیلی کوچک‌تر دیده می‌شود و کیفیت detection افت می‌کند.
# فقط وقتی crop خودرو از این آستانه بزرگ‌تر بود imgsz را متناسب بالا می‌بریم؛
# برای ماشین‌های معمولی (crop ≤ 640) رفتار قبلی دقیقاً حفظ می‌شود.
PLATE_IMGSZ_BASE = 640
PLATE_IMGSZ_MAX = 1280


def _plate_imgsz_for(v_crop):
    longest = max(v_crop.shape[:2])
    if longest <= PLATE_IMGSZ_BASE:
        return PLATE_IMGSZ_BASE
    return int(min(PLATE_IMGSZ_MAX, round(longest / 32) * 32))


def plate_consensus(reads):
    """لیست (متن، اطمینان، وزن) خوانش‌های یک ترک → (متن اجماعی، اطمینان میانگین).

    خوانش‌های یک پلاک معمولاً فقط در یک‌دو کاراکتر فرق دارند (O/0، D/0، M/H…)؛
    رای‌گیری روی *کل رشته* رأی‌ها را بین این نسخه‌های نزدیک تقسیم می‌کند. به‌جایش،
    مثل ALPRهای مرجع: اول طول غالب (وزن‌دار) انتخاب می‌شود، بعد برای هر جایگاه
    پرامتیازترین کاراکتر بین خوانش‌های هم‌طول. نتیجه می‌تواند رشته‌ای باشد که
    هیچ فریمی به‌تنهایی نخوانده — و همین هدف است.
    """
    by_len = Counter()
    for t, _, w in reads:
        by_len[len(t)] += w
    L = by_len.most_common(1)[0][0]
    same = [(t, c, w) for t, c, w in reads if len(t) == L]
    chars = []
    for i in range(L):
        col = Counter()
        for t, _, w in same:
            col[t[i]] += w
        chars.append(col.most_common(1)[0][0])
    return "".join(chars), float(np.mean([c for _, c, _ in same]))


def _match_track(tracks, cx, cy, plate_w, frame_i):
    """نزدیک‌ترین ترکِ زنده که در همین فریم هنوز پلاکی نگرفته؛ None = ترک جدید."""
    limit = max(MATCH_DIST_MIN, MATCH_DIST_PLATE_W * plate_w)
    best, dmin = None, limit
    for t in tracks:
        if t["last_frame"] == frame_i or frame_i - t["last_frame"] > MATCH_MAX_GAP:
            continue
        d = ((t["cx"] - cx) ** 2 + (t["cy"] - cy) ** 2) ** 0.5
        if d < dmin:
            dmin, best = d, t
    return best


def process_video(video_path, output_path, csv_path=None, max_frames=None):
    """ویدیوی ورودی → ویدیوی حاشیه‌دار + CSV یک ردیف برای هر پلاک (اجماع بین فریم‌ها).

    - هر وسیله‌ی نقلیه با نوعش برچسب می‌خورد (Car / Bus / Truck / Motorcycle / ...).
    - هر پلاک یک «ترک» است؛ همه‌ی خوانش‌های آن نگه داشته می‌شوند و متن نهایی با
      رای‌گیری کاراکتر‌به‌کاراکتر (plate_consensus) ساخته می‌شود. این از قفل‌شدن
      روی یک خوانشِ اشتباهِ تصادفاً پراطمینان (که با «فقط بهترین تک‌فریم» پیش
      می‌آمد) جلوگیری می‌کند و خطاهای تک‌کاراکتری فریم‌های مختلف همدیگر را
      خنثی می‌کنند.
    - هر فریم پردازش می‌شود — هیچ فریمی رد نمی‌شود.
    max_frames: فقط برای تست سریع؛ None یعنی کل ویدیو.
    """
    video_path, output_path = str(video_path), str(output_path)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"ویدیو باز نشد: {video_path}")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS)) or 25
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))

    tracks = []
    print(f"🚀 پردازش {Path(video_path).name} — {total} فریم (حالت {COUNTRY_MODE}, imgsz خودرو={IMGSZ})")
    frame_i = 0
    while True:
        ret, frame = cap.read()
        if not ret or (max_frames and frame_i >= max_frames):
            break
        frame_i += 1
        if frame_i % PROGRESS_EVERY == 0:
            print(f"⏳ فریم {frame_i}/{total}")

        # نسخه‌ی تمیز فریم: باکس/برچسب‌ها روی `frame` کشیده می‌شوند و اگر crop از
        # همان گرفته شود، خطوط رنگی داخل ورودی مدل پلاک و OCR می‌افتند.
        clean = frame.copy()
        v_res = vehicle_model.predict(clean, imgsz=IMGSZ, verbose=False, device=DEVICE)[0]
        for box in v_res.boxes:
            cls, conf = int(box.cls[0]), float(box.conf[0])
            if cls not in VEHICLE_CLASSES or conf < VEHICLE_CONF:
                continue
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            vname = VEHICLE_NAMES.get(cls, "Vehicle")
            draw_box(frame, x1, y1, x2, y2, (255, 150, 0))
            draw_plate_label(frame, x1, y1, f"{vname} {conf:.2f}", (255, 150, 0))

            v_crop = clean[y1:y2, x1:x2]
            if v_crop.size == 0 or cls in (1, 6):   # دوچرخه/قطار پلاک ندارند
                continue

            p_res = plate_model.predict(
                v_crop, imgsz=_plate_imgsz_for(v_crop), verbose=False, device=DEVICE
            )[0]
            for p in p_res.boxes:
                if float(p.conf[0]) < PLATE_CONF:
                    continue
                px1, py1, px2, py2 = map(int, p.xyxy[0])
                gbox = (x1 + px1, y1 + py1, x1 + px2, y1 + py2)
                gx1, gy1, gx2, gy2 = gbox
                cx, cy = (gx1 + gx2) // 2, (gy1 + gy2) // 2

                text, ocr_conf, script, weight = read_plate(clean, gbox)

                tr = _match_track(tracks, cx, cy, gx2 - gx1, frame_i)
                if tr is None:
                    tr = {"cx": cx, "cy": cy, "vehicle": vname, "text": "", "conf": -1.0,
                          "script": script, "info": None, "color": "", "frames": 0,
                          "last_frame": frame_i, "reads": [], "scripts": Counter()}
                    tracks.append(tr)
                tr["cx"], tr["cy"], tr["vehicle"] = cx, cy, vname
                tr["frames"], tr["last_frame"] = tr["frames"] + 1, frame_i

                if text:
                    tr["reads"].append((text, ocr_conf, weight))
                    tr["scripts"][script] += weight
                    best_text, tr["conf"] = plate_consensus(tr["reads"])
                    if best_text != tr["text"]:
                        plate_img = crop_plate(clean, gbox)
                        tr["text"] = best_text
                        tr["script"] = tr["scripts"].most_common(1)[0][0]
                        tr["color"] = detect_plate_color(plate_img)
                        tr["info"] = parse_iranian_plate(best_text, plate_img) if tr["script"] == "fa" else None

                draw_box(frame, gx1, gy1, gx2, gy2, (0, 255, 0))
                if tr["script"] == "fa" and tr["info"]:
                    label = ir_video_label(tr["info"])
                else:
                    label = latin_video_label(tr["text"])
                draw_plate_label(frame, gx1, gy1, label, (0, 255, 0))

        out.write(frame)

    cap.release()
    out.release()

    # --- CSV: یک ردیف برای هر پلاک (متن اجماعی) ---
    rows = []
    for i, tr in enumerate(tracks):
        if not tr["text"] or tr["frames"] < MIN_FRAMES:
            continue
        info = tr["info"]
        if tr["script"] == "fa" and info:
            rows.append([i, tr["vehicle"], "fa", info["raw"], info["digits"], info["letter"],
                         info["color"], info["type"], info["province"],
                         info["formatted"], round(tr["conf"], 3), tr["frames"]])
        else:
            rows.append([i, tr["vehicle"], tr["script"], tr["text"], "", "", tr["color"],
                         "", "", tr["text"], round(tr["conf"], 3), tr["frames"]])

    if csv_path:
        with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
            w = csv.writer(f)
            w.writerow(CSV_HEADER)
            w.writerows(rows)
        print(f"📄 CSV: {Path(csv_path).name}  ({len(rows)} پلاک یکتا)")

    print(f"✅ خروجی ویدیو: {output_path}")
    return output_path, rows


## بخش ۷ — تست روی ویدیوها

آدرس ویدیوهای تست را در `TEST_VIDEOS` بگذارید (مسیر کامل یا نام فایل داخل پوشه‌ی `ویدیو`).
برای هر ویدیو یک `<name>_out.mp4` و یک `<name>_plates.csv` در پوشه‌ی `خروجی` ساخته می‌شود.

In [49]:
TEST_VIDEOS = [
    # VIDEO_DIR / "1_P.mp4",
    # VIDEO_DIR / "2-1_P.mp4",
    # VIDEO_DIR / "2-2_P.mp4",
    # VIDEO_DIR / "5-2.mp4",
    VIDEO_DIR / "6-1.mp4",
]

results = []
for video in TEST_VIDEOS:
    video = Path(video)
    if not video.exists():
        print(f"⚠️ پیدا نشد: {video}")
        continue
    results.append(process_video(
        video,
        OUTPUT_DIR / f"{video.stem}_pelak3_out.mp4",
        OUTPUT_DIR / f"{video.stem}_pelak3_plates.csv",
        # max_frames=150,   # برای تست سریع؛ None برای کل ویدیو
    ))


In [50]:
from IPython.display import Video, display
import os
import subprocess

def convert_to_h264(input_path):
    output_path = input_path.replace('.mp4', '_h264.mp4')
    # استفاده از ffmpeg برای تبدیل کدک به فرمت قابل پخش در مرورگر
    command = f"ffmpeg -y -i {input_path} -c:v libx264 -crf 23 -preset fast -c:a aac -b:a 128k {output_path} -loglevel error"
    subprocess.run(command, shell=True)
    return output_path

# نمایش ویدیوهای پردازش شده با تبدیل کدک
if 'results' in locals() and results:
    for out_path, _ in results:
        if os.path.exists(out_path):
            print(f"🔄 در حال تبدیل کدک برای نمایش: {os.path.basename(out_path)}...")
            h264_path = convert_to_h264(str(out_path))
            print(f"🎬 در حال نمایش: {os.path.basename(h264_path)}")
            display(Video(h264_path, embed=True, width=800))
        else:
            print(f"⚠️ ویدیو در مسیر {out_path} یافت نشد.")
else:
    print("❌ هیچ نتیجه‌ای برای نمایش یافت نشد. ابتدا بخش پردازش ویدیو را اجرا کنید.")

In [51]:
خلاصه‌ی پلاک‌های یکتای هر ویدیو (قوی‌ترین خواندن هر پلاک)
import pandas as pd

for out_path, rows in results:
    print(f"\n🎥 {Path(out_path).name}  —  {len(rows)} پلاک یکتا")
    if not rows:
        continue
    df = pd.DataFrame(rows, columns=CSV_HEADER).sort_values("frames_seen", ascending=False)
    display(df[["vehicle", "script", "formatted", "type", "province", "color", "conf", "frames_seen"]])

In [52]:
# نمایش ویدیوی خروجی درون نوت‌بوک
# (اگر مرورگر کدک mp4v را پخش نکرد، فایل را مستقیم از پوشه‌ی خروجی باز کنید)
from IPython.display import Video, display

for out_path, _ in results:
    display(Video(str(out_path), embed=True, width=640))

## بخش ۸ — ساخت GIF از خروجی

از هر ویدیوی خروجی یک GIF کوتاه می‌سازد (برای اشتراک‌گذاری). در پوشه‌ی `خروجی` ذخیره می‌شود.

In [53]:
import imageio.v2 as imageio
from IPython.display import Image as IPyImage


def video_to_gif(mp4_path, gif_path, out_fps=8, scale=0.5, max_seconds=6):
    """ویدیو → GIF کوتاه و سبک."""
    cap = cv2.VideoCapture(str(mp4_path))
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 25
    step = max(1, round(src_fps / out_fps))
    max_frames = int(src_fps * max_seconds)

    frames, i = [], 0
    while True:
        ret, frame = cap.read()
        if not ret or i > max_frames:
            break
        if i % step == 0:
            if scale != 1:
                frame = cv2.resize(frame, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        i += 1
    cap.release()

    imageio.mimsave(str(gif_path), frames, fps=out_fps, loop=0)
    mb = Path(gif_path).stat().st_size / 1e6
    print(f"🎞️ {Path(gif_path).name}  —  {len(frames)} فریم، {mb:.1f} MB")
    return gif_path


for out_path, _ in results:
    gif = Path(out_path).with_suffix("").with_name(Path(out_path).stem + ".gif")
    video_to_gif(out_path, gif)
    display(IPyImage(filename=str(gif)))